# KeralaGAN: photo → Kerala mural, trained on Wikimedia Commons

Downloads openly licensed images (CC / public domain only, each logged in `sources.csv`), builds the dataset,
trains on the free T4 GPU and saves everything to Google Drive. **Runtime → Run all.**
If Colab disconnects, run all again: downloads are skipped and training resumes from Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/KeralaGAN'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
%cd /content
!test -d keralagan || git clone -q https://github.com/abhinavrenjithg-sys/keralagan.git
%cd /content/keralagan
!git pull -q
!pip install -q pytorch-fid

## 1. Download images from Wikimedia Commons

In [ ]:
import os, shutil
DATA_ZIP = f'{OUT}/datasets/mural.zip'
SOURCES = f'{OUT}/sources_mural.csv'
HAVE_DATA = os.path.exists(DATA_ZIP)
print('dataset already on Drive:', HAVE_DATA)
if not HAVE_DATA:
    # Domain A: Kerala performers and figures (mural paintings mostly show figures)
    !python scripts/fetch_commons.py --out /content/raw/people --dataset mural --domain A --sources {SOURCES} \
        --category "Kathakali" --category "Theyyam" --search "Kathakali" --search "Theyyam" \
        --search "Mohiniyattam" --search "Koodiyattam" --search "Kerala dancer" --limit 600
    # Domain B: Kerala mural paintings
    !python scripts/fetch_commons.py --out /content/raw/murals_full --dataset mural --domain B --sources {SOURCES} \
        --category "Kerala murals" --category "Mural paintings in Kerala" --depth 2 \
        --search "Kerala mural" --search "Kerala mural painting" --search "Mattancherry Palace mural" \
        --search "Krishnapuram Palace mural" --search "Kerala temple mural" --exclude logo --limit 400

## 2. Cut murals into patches, clean and split

In [ ]:
if not HAVE_DATA:
    !python scripts/make_patches.py --src /content/raw/murals_full --dst /content/raw/mural_patches --size 384 --stride 256 | tail -1
    !python scripts/prepare_dataset.py --a /content/raw/people --b /content/raw/mural_patches --name mural --root data
    os.makedirs(f'{OUT}/datasets', exist_ok=True)
    !cd data/mural && zip -qr {DATA_ZIP} .
else:
    !mkdir -p data/mural && unzip -q -o {DATA_ZIP} -d data/mural
for s in ['trainA','trainB','testA','testB']:
    print(s, len(os.listdir(f'data/mural/{s}')))

In [ ]:
# Contact sheet: top row domain A (photos), bottom row domain B (mural patches)
import random, glob
from PIL import Image
from IPython.display import display
random.seed(1)
def row(folder, n=8, s=128):
    files = random.sample(glob.glob(folder + '/*'), n)
    r = Image.new('RGB', (n*s, s))
    for i, f in enumerate(files):
        im = Image.open(f).convert('RGB'); w, h = im.size; m = min(w, h)
        r.paste(im.crop(((w-m)//2, (h-m)//2, (w+m)//2, (h+m)//2)).resize((s, s)), (i*s, 0))
    return r
sheet = Image.new('RGB', (1024, 256)); sheet.paste(row('data/mural/trainA'), (0, 0)); sheet.paste(row('data/mural/trainB'), (0, 128))
sheet.save(f'{OUT}/dataset_preview.png'); display(sheet)

## 3. Train (resumes automatically from Drive)

In [ ]:
EPOCHS = 60
!python -m keralagan.train --data data/mural --out {OUT} --epochs {EPOCHS} --amp --resume --workers 2

## 4. Results

In [ ]:
from IPython.display import Image as Show
samples = sorted(glob.glob(f'{OUT}/samples/mural/*.png'))
print(len(samples), 'epochs of samples'); display(Show(samples[-1]))

In [ ]:
!python scripts/evaluate_fid.py --data data/mural --ckpt {OUT}/checkpoints/mural/latest.pt --grid {OUT}/results/mural_grid.png
!cp results/mural_fid.json {OUT}/results/ 2>/dev/null
display(Show(f'{OUT}/results/mural_grid.png'))

In [ ]:
!python scripts/export_generator.py --ckpt {OUT}/checkpoints/mural/latest.pt --out {OUT}/weights/mural_G_AB.pt
print('Done. Weights, samples, FID and sources are in Drive → MyDrive/KeralaGAN')